In [2]:
import os
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\aviru\.vscode\RAG\RAG-With-langchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\aviru\AppData\Local\Temp\ipykernel_21220\983744508.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader


### PDF loader

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

## load all the pdf files from the directory
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", ##Pattern to match files
    loader_cls= PyMuPDFLoader, ##loader class to use
    show_progress=False
)
pdf_documents = dir_loader.load()

print(f"Loaded {len(pdf_documents)} pages")
print(pdf_documents[0].page_content)
print(pdf_documents[0].metadata)

### Split documents into chunks

In [4]:
def split_documents(documents, chunk_size=500, chunk_overlap=100):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)

    print(
        f"Split {len(documents)} documents into {len(split_docs)} chunks"
    )

    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [ ]:
chunks = split_documents(
    pdf_documents,
    chunk_size=500,
    chunk_overlap=100
)


In [ ]:
chunks

### embedding And vectorStoreDB

### Manage Embedding

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        Args:
           model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the Sentence Transformer model"""  
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
        
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
            Generate embeddings for a list of texts
            Args:
            texts: List of text strings to embed
            Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
           raise ValueError("Model not loaded")
       
        print (f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print (f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings        
    
    
## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1864.62it/s]


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [ ]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """             
            Initialize the vector store
            Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise    
        
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
            Add documents and their embeddings to the vector store
            Args:
                documents: List of LangChain documents
                embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
           raise ValueError("Number of documents must match number of embeddings")
       
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
        
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
            
            # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print (f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
        

vectorStore = VectorStore()
vectorStore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [ ]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings


['Avirup kar\navirupk57@gmail.com\n \nGitHub\n \nLinkedIn\n \nX.com\n \nPortfolio\n \nPROFILE\n-Full-stack developer building modern web applications with React and Next.js. I focus on clean UI, scalable \nbackend logic, and writing maintainable, performance-oriented code.\nI enjoy designing complete systems from frontend to backend while continuously improving my skills through \nreal-world projects.\nSKILLS\nFrontend & Backend Development\nReact.js, Next.js, Tailwind CSS, GSAP, Nest.js,',
 'SKILLS\nFrontend & Backend Development\nReact.js, Next.js, Tailwind CSS, GSAP, Nest.js, \nNode.js, Express.js, REST APIs, Next.js API routes, \nWebSocket\nLanguages / Programming Skills\nJavaScript (JS), TypeScript (TS)\nEventQueues\nRedis, RabbitMq\nDatabase(SQL/NoSQL)\nMongoDB , Postgresql, Redis\nVersion Control\nGit, GitHub\nPROJECTS\nAi-side-builder\u2060\n, This platform that helps you turn ideas into real websites in minutes',
 'PROJECTS\nAi-side-builder\u2060\n, This platform that helps yo